<a href="https://colab.research.google.com/github/pgmanuel/CAM_C04_NICE_Project/blob/pedro-dev/CAM_Project/src/test_CAM_NICE_PROJECT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Employer Project: National Institute for Health and Care Excellence (NICE)

For the Employer Project with the NICELinks to an external site., each group has been asked to build an AI-assisted method to recommend and justify clinical code lists as part of the pre-analysis phase for a given research question. This will enable NICE to better define the initial dataset more quickly and with higher confidence. They would like groups to use only publicly available materials.

The challenge is not to answer the question of whether or not an LLM can list codes. It’s to decipher whether a system can be designed to produce defensible, explainable, and reliable clinical code recommendations.’ ~ Shaun Rowark, Associate Director – Healthcare Data Analytics, NICE

Who is NICE?
The National Institute for Health and Care Excellence (NICE) exists to improve outcomes for people using the NHS and other public health and social care services by providing evidence-based guidance and advice. Its mission is to reduce variation in the quality of care across England and ensure that patients have access to effective, safe, and cost-efficient treatments.

NICE plays a central role in the English healthcare system by developing national guidance on the use of medicines, medical technologies, diagnostics, and clinical practices. One of its primary objectives is to assess the clinical and cost effectiveness of treatments to determine their suitability for use within the NHS. By evaluating both health outcomes and value for money, NICE supports the sustainable use of limited healthcare resources while maintaining high standards of care.

Internally, NICE undertakes a range of responsibilities to fulfil its role. These include commissioning and reviewing clinical evidence, conducting health economic analyses, consulting with healthcare professionals, patients, and industry stakeholders, and producing clear guidance for commissioners and practitioners. NICE also develops quality standards and performance indicators to support continuous improvement across health and social care services.

To fulfil these responsibilities, NICE maintains an internal healthcare data analytics function. This includes analysing healthcare data to develop real-world evidence, which will then be used to make decisions. In order to perform this analysis, different clinical terms need to be defined in the form of the relevant clinical codes.

The influence of NICE extends beyond England. NICE is internationally recognised as a leading authority in health technology assessment and evidence-based decision-making, and in the use of real-world evidence. Many countries and global health organisations refer to NICE’s methodologies and guidance as a benchmark for balancing innovation, affordability, and patient benefit within publicly funded healthcare systems.

To learn more about NICE and what it does, explore its website:

What NICE doesLinks to an external site.

[What NICE does](https://www.nice.org.uk/what-nice-does)

Scenario
The National Institute for Health and Care Excellence (NICE) supports evidence-based decision-making within the NHS by developing national guidance on the use of medicines, treatments, and healthcare interventions.

To produce high-impact guidance, NICE relies heavily on healthcare data analysis, particularly when assessing disease prevalence, treatment eligibility, and population-level impact. A critical component of this analysis involves defining clinical code sets, such as the Systematised Nomenclature of Medicine (SNOMED) or other diagnostic codes, which are used to identify patient cohorts within healthcare datasets.

To achieve this, NICE draws on a range of data sources, many of which are publicly available but complex to work with. These include NHS England reference sets, Quality and Outcomes Framework (QOF) – which specify clinical codes that should be used to adhere to a QOF indicator – and open clinical code repositories. The process of defining accurate and comprehensive clinical code lists for research questions is particularly challenging for two key reasons:


1.   The data sources are largely unstructured or semi-structured, often presented in formats such as spreadsheets, documents, or text-based repositories.
2.   The task requires significant domain expertise, as clinical conditions – especially those involving co-morbidities – can require hundreds of codes that must be carefully validated and justified, which can be subjective.

This project aims to improve the efficiency and accuracy of clinical code definition to support the pre-analysis phase of NICE workflows by applying advanced data science techniques, such as natural language processing (NLP) and large language models (LLMs), to these complex data sources. By enhancing NICE’s ability to generate and validate clinical code sets more effectively, the project aims to reduce manual effort, accelerate evidence generation, and ultimately improve the quality and consistency of NICE’s guidance and healthcare decision-making.

#Import data

In [ ]:
# 1. Completely wipe any trace of Pinecone
!pip uninstall -y pinecone pinecone-client --quiet

In [ ]:
# 2. Install the fresh, updated package bypassing the cache
!pip install --no-cache-dir --upgrade pinecone sentence-transformers pandas streamlit --quiet

In [ ]:
import os
from pinecone import Pinecone

# Initialize the Pinecone client
# (Remember to use your NEW API key!)
pc = Pinecone(api_key="pcsk_4ZbMjE_QZHYm2hAMpnoEhvAPFfgFH5Gp61SbC1hPx7JZwZEU9K8a9xzCCEmjEzaKWkW7HV")
index = pc.Index("nice")

print("Pinecone initialized successfully! We are back in business.")

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

##Import All Products

In [ ]:
products_data = pd.read_csv('/content/DAAR_2025_004_all_prod_codes.txt', sep='\t')


In [ ]:
# Display basic info
print(f"Dataset Shape: {products_data.shape}")
products_data.head()

In [ ]:
# 1. Extract Manufacturer (text inside parentheses at the end of the term)
products_data['manufacturer'] = products_data['term'].str.extract(r'\((.*?)\)')

In [ ]:
# 2. Identify Formulation Types (Tablet, Capsule, Solution, etc.)
def identify_formulation(term):
    term = term.lower()
    if 'tablet' in term: return 'Tablet'
    if 'capsule' in term: return 'Capsule'
    if 'suspension' in term or 'solution' in term: return 'Oral Liquid'
    if 'sachet' in term or 'powder' in term: return 'Powder/Sachet'
    if 'injection' in term: return 'Injection'
    return 'Other'

products_data['formulation_category'] = products_data['term'].apply(identify_formulation)

# 3. Identify special properties
products_data['is_sugar_free'] = products_data['term'].str.contains('sugar free', case=False)
products_data['is_modified_release'] = products_data['term'].str.contains('modified-release|XL|SR|MR|LA', case=False, regex=True)

print(f"Dataset Shape: {products_data.shape}")
products_data.head()

In [ ]:
print("\n--- Missing Values ---")
print(products_data.isnull().sum())

In [ ]:
prod_df = products_data.drop_duplicates(subset=['term'])

In [ ]:
products_data.to_csv('products_data.csv',index=False)

##Import Antihypertensive Codes

In [ ]:
# 1. Load the data (using tab separator based on your file format)
file_path = 'DAAR_2025_004_antihypertensive_codes.txt'
antihypertensive_data = pd.read_csv(file_path, sep='\t')

In [ ]:
# Clean up column names just in case there are hidden spaces
antihypertensive_data.columns = antihypertensive_data.columns.str.strip()

In [ ]:
# 2. Basic Information
print("--- Dataframe Info ---")
antihypertensive_data.info()

In [ ]:
print("\n--- Missing Values ---")
print(antihypertensive_data.isnull().sum())

In [ ]:
# 3. Exploratory Data Analysis (EDA): Top Drug Substances
top_drugs = antihypertensive_data['drugsubstancename'].value_counts().head(20)

print("\n--- Top 10 Drug Substances ---")
print(top_drugs)

In [ ]:
# 4. Visualize the Top 10
plt.figure(figsize=(10, 5))
top_drugs.plot(kind='bar', color='teal')
plt.title('Top 20 Most Common Antihypertensive Drug Substances')
plt.xlabel('Drug Substance')
plt.ylabel('Frequency')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
from sentence_transformers import SentenceTransformer

# 1. Load a fast, lightweight model to generate real vector embeddings
print("Loading AI model...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Clean the dataframe before uploading (drop rows without an ID or Term)
antihyper_df = antihypertensive_data.dropna(subset=['ProdCodeId', 'TermfromEMIS']).copy()

In [ ]:
vectors_to_upsert = []

print(f"Preparing {len(antihyper_df)} records for Pinecone...")

In [ ]:
# 3. Process each row
for _, row in antihyper_df.iterrows():
    # Pinecone requires IDs to be strings
    vector_id = str(row['ProdCodeId'])
    term = str(row['TermfromEMIS'])
    substance = str(row['drugsubstancename']) if pd.notna(row['drugsubstancename']) else "Unknown"

    # Generate the vector embedding from the medical term
    vector_embedding = model.encode(term).tolist()

    # Attach metadata so you can filter or read it later
    metadata = {
        "term": term,
        "drug_substance": substance
    }

    vectors_to_upsert.append((vector_id, vector_embedding, metadata))


In [ ]:
# 4. Upsert to Pinecone in batches of 100
batch_size = 100
for i in range(0, len(vectors_to_upsert), batch_size):
    batch = vectors_to_upsert[i : i + batch_size]
    index.upsert(vectors=batch)
    print(f"Upserted batch {i // batch_size + 1}")

print("\n All antihypertensive codes successfully loaded into Pinecone!")

Load the data from PINECONE

In [ ]:
# Check the actual stats of your database
stats = index.describe_index_stats()
print(stats)

In [ ]:
# 2. Create a dummy vector to trigger the search
# It must match the dimension size of the index (384)
dummy_vector = [0.1] * 384

print("Fetching data from Pinecone...")

In [ ]:
# 3. Query the index
# We use top_k=1000 to ensure we grab all your ~380 records
response = index.query(
    vector=dummy_vector,
    top_k=2000,        # Increased to easily cover the 1,192 records in default
    namespace="",      # "" targets the '__default__' namespace
    include_metadata=True,
    include_values=False # We usually leave the 384-number embeddings behind to save memory
)


In [ ]:
# 4. Parse the Pinecone response into a flat list
extracted_data = []

for match in response['matches']:
    # Build a dictionary for each row
    row_data = {
        "ProdCodeId": match['id'],
        "TermfromEMIS": match['metadata'].get('term', ''),
        "drugsubstancename": match['metadata'].get('drug_substance', '')
    }
    extracted_data.append(row_data)

In [ ]:
# 5. Convert to a Pandas DataFrame
df_pinecone = pd.DataFrame(extracted_data)

print(f"✅ Successfully loaded {len(df_pinecone)} records from Pinecone into a DataFrame!\n")

# Display the first 5 rows to verify
df_pinecone.head()

In [ ]:
print("\n--- Missing Values ---")
print(df_pinecone.isnull().sum())

Example of a Search Query

In [ ]:
# 3. Define your natural language search query
# Try changing this to things like "water pill", "beta blocker", or "medicine for fast heart rate"
search_query = "medication for severe swelling and fluid retention"

print(f"Searching for: '{search_query}'...\n")

In [ ]:
# 4. Convert the text query into a 384-dimension vector
query_vector = model.encode(search_query).tolist()

In [ ]:
# 5. Query Pinecone for the top 5 closest matches
response = index.query(
    vector=query_vector,
    top_k=5,
    namespace="", # Searching the default namespace where your data lives
    include_metadata=True
)

In [ ]:
# 6. Display the results nicely
print("--- Top 5 Semantic Matches ---")
for i, match in enumerate(response['matches']):
    score = match['score']
    med_name = match['metadata'].get('term', 'Unknown')
    substance = match['metadata'].get('drug_substance', 'Unknown')

    # The score indicates how closely the AI thinks the meaning matches (1.0 is a perfect match)
    print(f"{i+1}. Score: {score:.4f} | Substance: {substance}")
    print(f"   Term: {med_name}\n")

In [ ]:
# 2. Define the reusable function
def search_medications(user_query, top_results=5):
    """
    Searches the Pinecone vector database for medications matching the semantic meaning of the query.
    """
    print(f"🔍 Searching for: '{user_query}'...\n")

    try:
        # Convert the user's text into a vector
        query_vector = model.encode(user_query).tolist()

        # Query Pinecone
        response = index.query(
            vector=query_vector,
            top_k=top_results,
            namespace="", # Default namespace
            include_metadata=True
        )

        # Check if we got results
        if not response['matches']:
            print("No matches found.")
            return []

        # Format and display the results
        results = []
        for i, match in enumerate(response['matches']):
            score = match['score']
            med_name = match['metadata'].get('term', 'Unknown Term')
            substance = match['metadata'].get('drug_substance', 'Unknown Substance')
            code_id = match['id']

            # Print to the console for easy reading
            print(f"{i+1}. {med_name}")
            print(f"   Substance: {substance} | Match Score: {score:.4f} | ID: {code_id}\n")

            # Save to a list in case you want to use the data elsewhere in your code
            results.append({
                "term": med_name,
                "substance": substance,
                "score": score,
                "id": code_id
            })

        return results

    except Exception as e:
        print(f"An error occurred during the search: {e}")
        return []

print("✅ Search function ready to use!")

In [ ]:
# Try a symptom
print("--- Test 1 ---")
search_medications("pills for a racing heartbeat and anxiety", top_results=3)

# Try a broad category
print("--- Test 2 ---")
search_medications("calcium channel blockers", top_results=3)

# Try a specific dosage requirement
print("--- Test 3 ---")
search_medications("high dose ramipril capsules", top_results=3)